# Unidad 6 · Colab 2 de 3
## Introducción a la contenedorización: Docker

**Objetivos de este notebook**

- Entender qué es un contenedor y en qué se diferencia de una máquina virtual.
- Escribir un `Dockerfile` para una API/web app.
- Manejar los comandos esenciales de Docker (build, run, ps, logs, exec).
- Usar variables de entorno y volúmenes.
- Escribir un `docker-compose.yml` para levantar varios servicios juntos.
- Aplicar buenas prácticas (imágenes livianas, multi-stage builds, `.dockerignore`).

> **Nivel:** intermedio. Se asume que ya tenés una API o web app (de unidades anteriores) que querés contenedorizar.

> ⚠️ **Importante sobre Colab:** Google Colab **no tiene un demonio de Docker disponible**, así que los comandos `docker build` / `docker run` de este notebook **no se ejecutan acá**: son ejercicios para escribir y razonar sobre archivos de configuración. Al final del notebook te dejamos cómo instalar Docker en tu máquina para probarlos de verdad.

---

## 1. Contenedores vs. máquinas virtuales

Una **máquina virtual (VM)** virtualiza hardware completo y corre un sistema operativo entero por cada instancia (pesada, minutos para arrancar). Un **contenedor** empaqueta la aplicación con sus dependencias y comparte el kernel del sistema operativo anfitrión: es liviano, arranca en segundos y garantiza que *funciona igual en tu máquina, en CI y en producción*.

| | Máquina virtual | Contenedor |
|---|---|---|
| Aislamiento | Hardware virtualizado, SO completo | Proceso aislado, comparte el kernel del host |
| Peso | GBs | MBs |
| Arranque | Minutos | Segundos |
| Caso de uso típico | Aislar entornos muy distintos (SO diferente) | Empaquetar y distribuir una aplicación |

Documentación oficial: [¿Qué es un contenedor?](https://www.docker.com/resources/what-container/) · [Docker overview](https://docs.docker.com/get-started/docker-overview/)

## 2. El `Dockerfile`

Un `Dockerfile` es la receta para construir una imagen. Instrucciones más usadas:

| Instrucción | Qué hace |
|---|---|
| `FROM` | Imagen base de la que partimos |
| `WORKDIR` | Carpeta de trabajo dentro del contenedor |
| `COPY` | Copia archivos del host a la imagen |
| `RUN` | Ejecuta un comando durante el build (ej. instalar dependencias) |
| `ENV` | Define una variable de entorno |
| `EXPOSE` | Documenta qué puerto usa la app (informativo) |
| `CMD` | Comando que se ejecuta cuando arranca el contenedor |

Ejemplo para una API en FastAPI:

```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000
CMD uvicorn app.main:app --host 0.0.0.0 --port 8000
```

Documentación oficial: [Referencia del Dockerfile](https://docs.docker.com/reference/dockerfile/)

### Ejercicio 1 — Escribir un Dockerfile

Tenés una API en **Flask** (`app.py` en la raíz, entry point `app`), con dependencias en `requirements.txt`, que corre en el puerto `5000`. Escribí el `Dockerfile`.

```dockerfile
(completá acá)
```

<details>
<summary>💡 Ver solución</summary>

```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 5000
CMD python app.py
```

</details>

## 3. Comandos esenciales

```bash
docker build -t mi-api:1.0 .        # construye la imagen a partir del Dockerfile en el directorio actual
docker images                        # lista imágenes locales
docker run -d -p 8000:8000 --name mi-api mi-api:1.0   # corre un contenedor en segundo plano, mapea puertos
docker ps                            # contenedores corriendo
docker logs -f mi-api                # ver logs en vivo
docker exec -it mi-api bash          # abrir una terminal dentro del contenedor
docker stop mi-api                   # detener
docker rm mi-api                     # eliminar el contenedor
```

Documentación oficial: [Referencia de la CLI de Docker](https://docs.docker.com/reference/cli/docker/)

### Ejercicio 2 — Secuencia de comandos

Querés: construir la imagen `mi-api:1.0` desde el directorio actual, correrla mapeando el puerto `8000` del host al `80` del contenedor, ver sus logs, y luego entrar a una terminal dentro del contenedor. Escribí los comandos en orden.

```bash
(completá acá)
```

<details>
<summary>💡 Ver solución</summary>

```bash
docker build -t mi-api:1.0 .
docker run -d -p 8000:80 --name mi-api mi-api:1.0
docker logs -f mi-api
docker exec -it mi-api bash
```

</details>

## 4. Variables de entorno y volúmenes

- `-e CLAVE=valor` inyecta una variable de entorno; `--env-file .env` inyecta varias desde un archivo.
- `-v` monta un volumen para persistir datos (o reflejar código local dentro del contenedor).

```bash
docker run -d --env-file .env -v datos_db:/var/lib/postgresql/data postgres:16
```

Documentación oficial: [Variables de entorno en Compose](https://docs.docker.com/compose/how-tos/environment-variables/set-environment-variables/) · [Volúmenes](https://docs.docker.com/engine/storage/volumes/)

### Ejercicio 3 — Env vars y volumen persistente

Corré un contenedor de **Postgres 16** con usuario `admin`, contraseña `secreto123`, base de datos `tareas_db`, y un volumen llamado `tareas_data` para que los datos no se pierdan si el contenedor se reinicia.

```bash
(completá acá)
```

<details>
<summary>💡 Ver solución</summary>

```bash
docker run -d \
  --name db-tareas \
  -e POSTGRES_USER=admin \
  -e POSTGRES_PASSWORD=secreto123 \
  -e POSTGRES_DB=tareas_db \
  -v tareas_data:/var/lib/postgresql/data \
  postgres:16
```

</details>

## 5. `.dockerignore`

Evita copiar archivos innecesarios (o sensibles) a la imagen, acelerando el build y reduciendo su tamaño.

```dockerignore
__pycache__/
*.pyc
.venv/
.git/
.env
node_modules/
Dockerfile
.dockerignore
```

Documentación oficial: [Contexto de build y .dockerignore](https://docs.docker.com/build/concepts/context/#dockerignore-files)

### Ejercicio 4 — Escribir un `.dockerignore`

Para la API de tareas (Python + Git + entorno virtual), escribí un `.dockerignore` que evite copiar archivos innecesarios a la imagen.

```dockerignore
(completá acá)
```

<details>
<summary>💡 Ver solución</summary>

```dockerignore
__pycache__/
*.pyc
.venv/
env/
.git/
.gitignore
.env
*.md
tests/
```

</details>

## 6. `docker-compose` básico

Cuando la app necesita varios servicios (por ejemplo, API + base de datos), `docker-compose` permite definirlos y levantarlos juntos con un solo comando.

```yaml
services:
  api:
    build: .
    ports:
      - '8000:8000'
    environment:
      - REDIS_HOST=redis
    depends_on:
      - redis

  redis:
    image: redis:7-alpine
    ports:
      - '6379:6379'
```

```bash
docker compose up -d     # levanta todos los servicios
docker compose logs -f   # ver logs de todos
docker compose down      # apagar y limpiar
```

Documentación oficial: [Docker Compose](https://docs.docker.com/compose/)

### Ejercicio 5 — `docker-compose.yml`

Tu API en **Flask** necesita conectarse a **Redis** para cachear resultados. Escribí un `docker-compose.yml` con dos servicios: `api` (buildeado desde el `Dockerfile` local, puerto `5000`) y `redis` (imagen oficial `redis:7-alpine`).

```yaml
(completá acá)
```

<details>
<summary>💡 Ver solución</summary>

```yaml
services:
  api:
    build: .
    ports:
      - '5000:5000'
    environment:
      - REDIS_HOST=redis
    depends_on:
      - redis

  redis:
    image: redis:7-alpine
```

</details>

## 7. Buenas prácticas

- Usar imágenes base livianas (`slim`, `alpine`) cuando sea posible.
- Copiar primero `requirements.txt`/`package.json` e instalar dependencias **antes** de copiar el resto del código, para aprovechar el cache de capas de Docker.
- No correr el proceso como usuario `root` dentro del contenedor.
- Usar **multi-stage builds** para que la imagen final no incluya herramientas de compilación.
- Fijar versiones de la imagen base (`python:3.11-slim`, no `python:latest`).

Documentación oficial: [Buenas prácticas para escribir Dockerfiles](https://docs.docker.com/build/building/best-practices/)

### Ejercicio 6 (avanzado) — Optimizar un Dockerfile

Este `Dockerfile` funciona, pero tiene varios problemas de buenas prácticas. Identificalos y reescribilo.

```dockerfile
FROM python:latest
COPY . /app
WORKDIR /app
RUN pip install -r requirements.txt
CMD python app.py
```

<details>
<summary>💡 Ver solución</summary>

**Problemas:** usa `python:latest` (no reproducible), copia todo el código antes de instalar dependencias (invalida el cache en cada cambio de código), y corre como `root`.

```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

RUN useradd -m appuser
USER appuser

CMD python app.py
```

</details>

## Mini-proyecto: dockerizar tu API

1. Escribí el `Dockerfile` y `.dockerignore` para la API/web app que venís construyendo en el curso.
2. Si depende de una base de datos u otro servicio, agregá un `docker-compose.yml`.
3. Instalá [Docker Desktop](https://docs.docker.com/desktop/) en tu máquina (fuera de Colab) para probarlo:

```bash
docker build -t mi-api:1.0 .
docker run -d -p 8000:8000 --env-file .env mi-api:1.0
curl http://localhost:8000
```

4. Confirmá que la API responde y revisá los logs con `docker logs`.

**Entregable:** `Dockerfile` (+ `.dockerignore` y `docker-compose.yml` si aplica) subidos a tu repositorio de GitHub, y una captura o video corto mostrando el contenedor corriendo.

---

**Seguís en:** *Colab 3 — Despliegue en PaaS: Render, Railway y Hugging Face Spaces*